# Canonical Model 07: Transport, Particle Tracking, and Calibration

This notebook adds **solute transport** to the canonical valley model and calibrates it
against **concentration** data.

That last part is the point. Heads are almost insensitive to porosity — in fact
completely insensitive, since porosity does not appear in the flow equation — and only
weakly constrain where water actually goes; a plume is sensitive to both. So here the
observations include concentrations at monitoring wells, and what they recover is
**hydraulic conductivity and porosity together**: flow and transport properties
inferred from the shape and timing of a contaminant plume. Along the way we release
particles from the source to see the advective picture the plume is painting.

In [ ]:
import sys
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import pandas as pd
import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6.canonical_transport import (
    build_canonical_transport_calibration_demo,
)

notebook_header('07', 'Transport, Particle Tracking, and Calibration',
                'Attach GWT to the canonical model, track particles, and calibrate K '
                'against concentrations.')

## 1. Build the transport model as synthetic truth

`build_canonical_transport_calibration_demo` builds the canonical valley model with a
**GWT sibling attached to the same simulation**, puts a constant-concentration source
across the up-valley face, runs it, samples heads and concentrations at downgradient
monitoring wells as the "measured" data, then spoils both hydraulic conductivity and
porosity so there is real work for history matching to do.

The transport model is a *sibling on one flat simulation*, not a separate project. That
is deliberate: it keeps both models' external arrays in one directory, which is where
PEST looks for the parameter files it multiplies.

**What to look for:** a plume that has spread well beyond its source cells, and
monitoring wells sitting at intermediate concentrations — a well pinned at the source
value cannot constrain anything.

In [ ]:
import shutil
artifact_root = Path('../artifacts/canonical_transport')
shutil.rmtree(artifact_root, ignore_errors=True)  # clean rebuild
artifact_root.mkdir(parents=True, exist_ok=True)

# `validation()` (50x50) is the honest profile for this notebook; `testing()` is
# faster if you are just reading along.
config = mf.CanonicalModelConfig.validation()
demo = build_canonical_transport_calibration_demo(artifact_root / 'model', config=config)

transport = demo.transport
view = transport.transport_view()   # the GWT sibling, read through the same grammar
conc = view.conc.get()

pd.Series({
    'flow_model': demo.model.name,
    'transport_model': transport.transport_name,
    'source_cells': len(transport.source_cells),
    'monitoring_wells': len(demo.well_cells),
    'forecast_cell': demo.forecast_cell,
    'max_concentration': round(float(conc['conc'].max()), 4),
    'cells_above_1pct': int((conc['conc'] > 0.01).sum()),
    'start_K_factor': demo.start_k_factor,
    'start_porosity_factor': demo.start_porosity_factor,
}, name='transport model')

## 2. Read the plume

`model.conc` is the transport twin of `model.hds` and answers the same verbs —
`get`/`summary`/`array`/`map`/`xs`/`mosaic`/`animate`. Nothing about the reading
grammar changes because the physics did.

**What to look for:** concentration highest along the up-valley source and drawn down
the valley axis, following the same high-K paleochannel the head maps show.

In [ ]:
view.conc.map(per=config.nper - 1, layer=0).plot()

## 3. The transport budget

`model.budget.<term>` exposes every term in a model's own budget file as a spatial
noun. The terms are **discovered from the file**, so a transport model shows its own
vocabulary — `source_sink_mix`, `storage_aqueous`, and one term per advanced package.

**What to look for:** the mass entering through `cnc` (the source) balanced against
storage and the advanced-package terms. `flow_ja_face` is listed but refuses `.get()`:
it is indexed by cell connection, not by cell, so it is not a per-cell table.

In [ ]:
print('budget terms:', view.budget.types)

# the source term, per cell, as a table and a map
source_flux = view.budget.cnc.get(per=config.nper - 1)
display(view.budget.cnc.summary())
view.budget.cnc.map(per=config.nper - 1).plot()

## 4. Where does the source water go? (PRT)

Particles released from the contaminant source cells trace the advective paths the
plume follows. This is the same MF6 PRT engine notebook 03 uses, run against the flow
model of the coupled simulation.

**What to look for:** particle tracks that overlay the plume from section 2 — the
pathlines are the skeleton, the plume is the same story with dispersion added.

In [ ]:
releases = mf.PRTReleasePoints.from_cells(
    demo.model, list(transport.source_cells), layer=0, group='source'
)
prt = demo.model.particle_tracking.prt(
    workspace=artifact_root / 'prt',
    release_points=releases,
    porosity=0.25,
    extend_tracking=False,
)
prt_results = prt.run(silent=True)
assert prt_results.success

tracks = prt_results.pathlines.get()
pd.Series({
    'released_particles': len(releases.packagedata),
    'pathline_records': len(tracks),
    'tracked_particles': tracks['particle'].nunique(),
    'max_travel_time': round(float(prt_results.pathlines.summary()['travel_time'].max()), 1),
}, name='PRT results')

In [ ]:
# One polyline per particle over the final-period water table.
prt_results.pathlines.map(
    per=config.nper - 1,
    layer=0,
    title='Particle paths from the contaminant source',
).plot()

Two derived views summarize the same run per cell. Both are **time-integrated** over
the whole tracking run, so neither takes `per=`.

**What to look for:** travel time increasing down the valley, and endpoints clustering
where the flow system discharges — the stream corridor and the lake.

In [ ]:
display(prt_results.travel_time.map(stat='median').plot())
display(prt_results.endpoints.plot(top=15))

## 5. Declare the calibration

Now the interesting part. We estimate a flow property (**hydraulic conductivity**) and
a transport property (**porosity**) together — and this is where the two data types
earn their place.

`ConcTargets` is the concentration twin of `HeadTargets`: same point-sampling
semantics, reading the transport model's `.ucn` instead of the flow model's `.hds`.
`cal.observe(...)` takes either directly.

**Why both heads and concentration?** Porosity does not appear in the flow equation at
all, so heads cannot constrain it — measured on this model, a 3x porosity change moves
heads by *exactly* 0.000000 ft. Concentration does respond to it. But transport
velocity is `v = Ki/n`, so raising K and lowering porosity move the plume almost
identically: the concentration responses to `K x3` and `porosity /3` have **cosine
similarity 0.98** at these wells. Estimating both from concentration alone is
therefore ill-posed — the calibration can trade one against the other at no cost in
phi. Heads break the tie: they pin K, and concentration then pins porosity.

`cal.parameterize('porosity')` finds the GWT sibling on its own; the target knows
which model owns the file.

**What to look for:** two adjustable parameters, and a nonzero-observation count of
wells x stress periods x *two* data types.

In [ ]:
cal = demo.model.pest('transport_pest', start_datetime='2024-01-01')
cal.parameterize('k', style='constant', bounds=(0.05, 2.0), physical=(0.01, 300.0), capture=True)
# Porosity lives on the GWT sibling; the target resolves it. `physical=` sits above
# the model's own value on purpose -- it clamps the FINAL porosity, so a bound at the
# starting value would swallow every multiplier above 1.0.
cal.parameterize('porosity', style='constant', bounds=(0.3, 3.0), physical=(0.02, 0.6), capture=True)

cal.observe(demo.head_targets)        # pins K -- porosity is invisible to heads
cal.observe(demo.conc_targets)        # pins porosity, given K
cal.forecast(demo.forecast_targets)   # a downgradient concentration prediction
pst = cal.build('transport_pest.pst', noptmax=0)

# The template is a SIBLING of the model directory (<model>.pest/<name>), not a
# child, so PstFrom's copy of the model workspace cannot swallow it -- this cell
# is safe to re-run on its own, and several named calibrations coexist.
pd.Series({
    'template_workspace': cal.template_workspace.name,
    'adjustable_parameters': pst.npar_adj,
    'nonzero_observations': pst.nnz_obs,
    'observation_wells': len(demo.well_cells),
    'stress_periods': config.nper,
    'start_K_factor': demo.start_k_factor,
    'start_porosity_factor': demo.start_porosity_factor,
    'any_NaN_observations': bool(pst.observation_data['obsval'].isna().any()),
}, name='calibration problem')

## 6. Validate the forward run

Every PEST iteration calls `forward_run.py`. Running it once directly is the key setup
check: it must apply the parameter multipliers, run MODFLOW, and regenerate the
simulated-concentration file that pyEMU's instruction file reads.

**What to look for:** a zero return code and a regenerated `conc_simulated_conc.csv`
whose values are not all zero (all-zero means the wrong binary was read).

In [ ]:
import subprocess
template = cal.template_workspace
result = subprocess.run([sys.executable, 'forward_run.py'], cwd=template,
                        capture_output=True, text=True)
assert result.returncode == 0, result.stdout + '\n' + result.stderr

simulated = pd.read_csv(template / 'conc_simulated_conc.csv')
pd.Series({
    'forward_run_returncode': result.returncode,
    'reads_the_transport_ucn': transport.transport_name in (template / 'forward_run.py').read_text(),
    'simulated_conc_regenerated': True,
    'simulated_rows': len(simulated),
    'values_nonzero': bool((simulated.drop(columns=['per']).abs().to_numpy() > 1e-9).any()),
}, name='forward-run validation')

## 7. Run PESTPP-IES

PESTPP-IES fires hundreds of forward solves and each one runs both the flow and the
transport model — this is a go-get-coffee cell. It is gated off by default; set
`RUN_IES = True` to regenerate.

**What to look for:** phi dropping between iterations, and the posterior ensemble
bracketing the measured concentrations more tightly than the prior did.

In [ ]:
# Set RUN_IES = True to actually run. Each solve runs GWF *and* GWT, so this is
# slower per realization than the flow-only calibration in notebook 04.
RUN_IES = True
REALS = 30
ITERATIONS = 3
WORKERS = 8

ies = None
if RUN_IES:
    ies = cal.run_ies(reals=REALS, iterations=ITERATIONS, workers=WORKERS)
ies

In [ ]:
if ies is not None:
    display(ies.plot_phi())        # phi dropping between iterations
    display(ies.plot_vs_obs())     # posterior ensemble vs the measured values
    display(ies.forecasts())

    # The calibrated fields. These are ENSEMBLE statistics, not one realization:
    # `std` and `reduction` are what a single deterministic run cannot tell you.
    display(ies.plot_field('k', stat='mean', which='posterior', layer=0))
    display(ies.plot_field('k', stat='reduction', layer=0))
    display(ies.plot_field('porosity', stat='mean', which='posterior', layer=0))

    # Where is the model still biased? One family at a time -- `prefix=` is the
    # observation-set prefix ('hds' for heads, 'conc' for concentration). Heads are a
    # length and concentration a mass per volume, and one shared color scale would let
    # the larger-magnitude family set the limit and render the other uniformly white,
    # which reads as a perfect fit. Without `prefix=` this warns and draws both.
    display(ies.plot_obs_residuals(prefix='hds'))
    display(ies.plot_obs_residuals(prefix='conc'))

    # `best()` returns the 'base' realization -- the minimum-error-variance
    # parameter set. Prefer it to criterion='min_phi', which tends to be over-fit.
    print('best realization:', ies.best())

## Interpretation checklist

- **The plume is where the flow field says it should be.** Section 2's concentration
  map and section 4's particle tracks should tell the same story; if they disagree,
  suspect the transport source or the porosity, not the flow solution.
- **The monitoring wells can actually move.** A well pinned at the source
  concentration constrains nothing — `monitoring_well_cells` keeps intermediate-value
  cells downgradient of the source for exactly this reason.
- **No NaN observations.** Section 5 checks this explicitly: a control file full of
  NaN observations builds and runs and calibrates to nothing.
- **The forward run reads the transport binary.** Section 6 asserts the GWT model's
  name appears in `forward_run.py`; reading the flow model's `.hds` instead would
  produce plausible-looking numbers of entirely the wrong quantity.
- **K is identifiable from concentration here.** On the canonical model a 3x K change
  moves concentration at most monitoring wells by more than 1%. That is what makes
  this calibration well posed rather than decorative.
- **K and porosity are separated by the DATA, not by the algorithm.** From
  concentration alone they are near-collinear (`v = Ki/n`; measured cosine similarity
  0.98), so a concentration-only run can drive both far from the truth while phi looks
  fine. Check that both `plot_field` maps land near their true values, not just that
  phi dropped — if porosity is recovered but K is not (or vice versa, in compensating
  directions), the head targets are doing less work than they should.

Continue to **04 · PEST Calibration and Results** for the flow-only calibration this
one is modelled on, or **03 · PRT and Parallel** for particle tracking at scale.

In [ ]:
from myflopy.modflow.mf6.pest.ies import IesResults

ies: IesResults

ies.plot_field('k', stat='mean', which='posterior', layer=0).show()